# pipelines.extractor.base

> Pipline for information extraction

In [ ]:
# | default_exp pipelines.extractor.base

In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | export

import os
from typing import List, Optional, Callable, Union, Any, Set
import pandas as pd
import json
from onprem.utils import segment

from onprem.ingest import load_single_document
from onprem.ingest.helpers import extract_extension

try:
    from pydantic import BaseModel
except ImportError:
    BaseModel = None


class Extractor:
    def __init__(
        self,
        llm, # An `onprem.LLM` object
        prompt_template: Optional[str] = None, # A model specific prompt_template with a single placeholder named "{prompt}". If supplied, overrides the `prompt_template` supplied to the `LLM` constructor.
        **kwargs,
    ):
        """
        `Extractor` applies a given prompt to each sentence or paragraph in a document and returns the results.
        """
        self.llm = llm
        self.prompt_template = prompt_template if prompt_template is not None else llm.prompt_template



    def apply(self,
              ex_prompt_template:str, # A prompt to apply to each `unit` in document. Should have a single variable, `{text}`
              fpath: Optional[str] = None, # A path to to a single file of interest (e.g., a PDF or MS Word document). Mutually-exclusive with `content`.
              content: Optional[str] = None, # Text content of a document of interest.  Mutually-exclusive with `fpath`.
              unit:str='paragraph', # One of {'sentence', 'paragraph'}. 
              pydantic_model=None, # If a Pydantic model is supplied, `LLM.pydantic_prompt` is used instead of `LLM.prompt`.
              attempt_fix:bool=False, # If True and `pydantic_model` is supplied, attempt to fix malformed/incomplete outputs.
              fix_llm=None, # LLM used to attempt fix when `attempt_fix=True`. If None, then use `self.llm.llm`.
              preproc_fn: Optional[Callable] = None, # Function should accept a text string and returns a new preprocessed input.
              filter_fn: Optional[Callable] = None, # A function that accepts a sentence or paragraph and returns `True` if prompt should be applied to it.
              clean_fn: Optional[Callable] = None, # A function that accepts a sentence or paragraph and returns "cleaned" version of the text. (applied after `filter_fn`)
              pdf_pages:List[int]=[], # If `fpath` is a PDF document, only apply prompt to text on page numbers listed in `pdf_pages` (starts at 1).
              maxchars = 2048, # units (i.e., paragraphs or sentences) larger than `maxchars` split.
              stop:list=[], # list of characters to trigger the LLM to stop generating.
              **kwargs, # Extra kwargs are fed to `load_single_document`
             ):
        """
        Apply the prompt to each `unit` (where a "unit" is either a paragraph or sentence) optionally filtered by `filter_fn`.
        Extra kwargs fed directly to `load_single_document`.
        Results are stored in a `pandas.Dataframe`.
        """
        if not(bool(fpath) != bool(content)):
            raise ValueError('Either fpath argument or content argument must be supplied but not both.')
        if pdf_pages and kwargs.get('pdf_unstructured', False):
            raise ValueError('The parameters pdf_pages and pdf_unstructured are mutually exclusive.')
        if pdf_pages and kwargs.get('pdf_markdown', False):
            raise ValueError('The parameters pdf_pages and pdf_markdown are mutually exclusive.')
            
        # setup extraction prompt
        extraction_prompt = ex_prompt_template if self.prompt_template is None else self.prompt_template.format(**{'prompt': ex_prompt_template})   

        # extract text
        if not content:
            if not os.path.isfile(fpath):
                raise ValueError(f'{fpath} is not a file')
            docs = load_single_document(fpath, **kwargs)
            if not docs: return
            ext = extract_extension(fpath)
            if ext == 'pdf' and pdf_pages:
                docs = [doc for i,doc in enumerate(docs) if i+1 in pdf_pages]
            content = '\n\n'.join([preproc_fn(doc.page_content) if preproc_fn else doc.page_content for doc in docs])
        
        # segment
        chunks = segment(content, maxchars=maxchars, unit=unit)
        extractions = []
        texts = []
        for chunk in chunks:
            if filter_fn and not filter_fn(chunk): continue
            if clean_fn: chunk = clean_fn(chunk)
            prompt = extraction_prompt.format(text=chunk)
            if pydantic_model:
                fix_llm = fix_llm if fix_llm else self.llm.llm
                output = self.llm.pydantic_prompt(prompt, pydantic_model,
                                                  attempt_fix=attempt_fix,
                                                  fix_llm=fix_llm,
                                                  stop=stop)
                output = output if isinstance(output, str) else output.model_dump_json()
                extractions.append(output)
            else:
                extractions.append(self.llm.prompt(prompt, stop=stop))
            texts.append(chunk)
        df = pd.DataFrame({'Extractions':extractions, 'Texts':texts})
        return df


    def extract_structured(self,
                          prompt:str, # Prompt for structured extraction (should include {text} placeholder if needed)
                          pydantic_model, # Pydantic model defining the output structure
                          fpath: Optional[str] = None, # Path to a single file. Mutually-exclusive with `content`.
                          content: Optional[str] = None, # Text content. Mutually-exclusive with `fpath`.
                          max_length: Optional[int] = None, # Maximum characters to process from document (None = no limit)
                          preproc_fn: Optional[Callable] = None, # Preprocessing function for document text
                          pdf_pages:List[int]=[], # If `fpath` is a PDF, only use text from these page numbers
                          return_as: str = 'model', # One of {'model', 'dict', 'json'}. Format to return results.
                          stop:list=[], # List of stop sequences for LLM
                          filter_fn: Optional[Callable[[Any], bool]] = None, # Filter function for extracted items (applied after extraction)
                          postproc_fn: Optional[Callable[[Any], Any]] = None, # Post-processing function (applied after filtering)
                          filter_field: Optional[str] = None, # Field name to filter (auto-detect if None)
                          **kwargs, # Extra kwargs fed to `load_single_document`
                         ) -> Union[BaseModel, dict, str]:
        """
        Perform document-level structured extraction using Pydantic models.
        
        This method processes the entire document (or a large section) in a single LLM call,
        using a Pydantic model to enforce structured output. This is more efficient than
        chunk-by-chunk processing when you need to extract structured information that may
        span multiple sections of the document.
        
        Unlike `apply()` which processes each paragraph/sentence separately, this method:
        - Makes a single LLM call for the entire document
        - Maintains full document context for better relationship detection
        - Enforces schema compliance via Pydantic models
        - Returns structured data in your desired format
        
        Example use cases:
        - Extract all system parameters with values and units
        - Extract key performance parameters with thresholds and objectives
        - Extract entities, relationships, or structured specifications
        - Any extraction requiring cross-document context
        
        Args:
            prompt: Prompt template for extraction. Use {text} placeholder if you want 
                   the document text inserted (recommended for clarity).
            pydantic_model: Pydantic BaseModel class defining output structure
            fpath: Path to document file (PDF, Word, text, etc.)
            content: Raw text content (alternative to fpath)
            max_length: Truncate document to this many characters (None = use full document)
            preproc_fn: Optional function to preprocess document text before extraction
            pdf_pages: For PDFs, extract only from these page numbers (1-indexed)
            return_as: Output format - 'model' (Pydantic model), 'dict', or 'json' string
            stop: Stop sequences for LLM generation
            **kwargs: Additional arguments passed to load_single_document
            
        Returns:
            Extracted data in format specified by return_as parameter:
            - 'model': Pydantic model instance
            - 'dict': Python dictionary
            - 'json': JSON string
            
        Example:
            ```python
            from pydantic import BaseModel, Field
            from typing import List
            
            class Parameter(BaseModel):
                name: str = Field(description="Parameter name")
                value: float = Field(description="Numeric value")
                unit: str = Field(description="Unit of measurement")
                
            class ParameterList(BaseModel):
                parameters: List[Parameter] = Field(default=[], description="All parameters found")
            
            extractor = Extractor(llm)
            result = extractor.extract_structured(
                prompt="Extract all technical parameters from: {text}",
                pydantic_model=ParameterList,
                fpath="specs.pdf",
                return_as='dict'
            )
            print(result['parameters'])
            ```
        """
        if BaseModel is None:
            raise ImportError("Pydantic is required for structured extraction. Install with: pip install pydantic")
        
        if not(bool(fpath) != bool(content)):
            raise ValueError('Either fpath argument or content argument must be supplied but not both.')
        if pdf_pages and kwargs.get('pdf_unstructured', False):
            raise ValueError('The parameters pdf_pages and pdf_unstructured are mutually exclusive.')
        if pdf_pages and kwargs.get('pdf_markdown', False):
            raise ValueError('The parameters pdf_pages and pdf_markdown are mutually exclusive.')
        if return_as not in ['model', 'dict', 'json']:
            raise ValueError("return_as must be one of: 'model', 'dict', or 'json'")
            
        # Extract text from document
        if not content:
            if not os.path.isfile(fpath):
                raise ValueError(f'{fpath} is not a file')
            docs = load_single_document(fpath, **kwargs)
            if not docs: 
                raise ValueError(f'No content loaded from {fpath}')
            ext = extract_extension(fpath)
            if ext == 'pdf' and pdf_pages:
                docs = [doc for i,doc in enumerate(docs) if i+1 in pdf_pages]
            content = '\n\n'.join([preproc_fn(doc.page_content) if preproc_fn else doc.page_content for doc in docs])
        
        # Apply preprocessing if provided
        if preproc_fn and content:
            content = preproc_fn(content)
            
        # Truncate if max_length specified
        if max_length is not None:
            content = content[:max_length]
        
        # Format the prompt with the text if {text} placeholder exists
        if '{text}' in prompt:
            final_prompt = prompt.format(text=content)
        else:
            # If no placeholder, append the content
            final_prompt = f"{prompt}\n\n{content}"
        
        # Prepare response_format - automatically convert to vLLM format if needed
        response_format = self._prepare_response_format(pydantic_model)
        is_vllm_format = isinstance(response_format, dict)
        
        # Make the LLM call with structured output
        try:
            response = self.llm.prompt(final_prompt, response_format=response_format, stop=stop)
        except Exception as e:
            raise RuntimeError(f"Failed to extract structured data: {e}")
        
        # Convert response to Pydantic model if vLLM format was used
        if is_vllm_format:
            if isinstance(response, str):
                # Parse JSON string response
                response = pydantic_model.model_validate_json(response)
            elif isinstance(response, dict):
                # Convert dict to Pydantic model
                response = pydantic_model.model_validate(response)
        
        # Apply post-processing BEFORE format conversion
        if postproc_fn:
            response = postproc_fn(response)
        
        if filter_fn:
            response = self._apply_filter_fn(response, filter_fn, filter_field)
        
        # Return in requested format
        if return_as == 'model':
            return response
        elif return_as == 'dict':
            return response.model_dump() if hasattr(response, 'model_dump') else response.dict()
        else:  # return_as == 'json'
            return response.model_dump_json() if hasattr(response, 'model_dump_json') else response.json()
    
    
    def _prepare_response_format(self, pydantic_model):
        """
        Prepare response_format for LLM call.
        
        For local APIs (vLLM, OpenLLM, etc.), automatically convert Pydantic model 
        to vLLM's json_schema dict format for better compatibility.
        For other backends (OpenAI, Anthropic, etc.), pass through the Pydantic 
        model to use LangChain's native structured output support.
        
        Args:
            pydantic_model: Pydantic BaseModel class
            
        Returns:
            Either dict (for vLLM) or Pydantic model (for other backends)
        """
        if self.llm.is_local_api():
            # Convert to vLLM json_schema format
            # See: https://amaiya.github.io/onprem/examples_guided_prompts.html
            return {
                "type": "json_schema",
                "json_schema": {
                    "name": pydantic_model.__name__.lower(),
                    "schema": pydantic_model.model_json_schema()
                }
            }
        else:
            # Pass through Pydantic model for LangChain's native support
            return pydantic_model
    
    
    def _get_list_fields(self, model: BaseModel) -> List[str]:
        """Automatically detect List[T] fields in Pydantic model."""
        from typing import get_origin
        
        list_fields = []
        for field_name, field_info in model.model_fields.items():
            origin = get_origin(field_info.annotation)
            if origin is list:
                list_fields.append(field_name)
        
        return list_fields
    
    
    def _apply_filter_fn(
        self,
        model: BaseModel,
        filter_fn: Callable[[Any], bool],
        filter_field: Optional[str] = None
    ) -> BaseModel:
        """
        Apply filter_fn to list field(s) in the Pydantic model.
        
        Args:
            model: The Pydantic model instance
            filter_fn: Function that takes an item and returns True to keep it
            filter_field: Specific field name to filter (None = auto-detect)
        
        Returns:
            New model instance with filtered data
        """
        import warnings
        
        # Get all list fields or specific field
        if filter_field:
            list_fields = [filter_field] if filter_field in model.model_fields else []
        else:
            list_fields = self._get_list_fields(model)
        
        if not list_fields:
            warnings.warn(
                f"No list fields found in {model.__class__.__name__}. "
                f"filter_fn has no effect. Available fields: {list(model.model_fields.keys())}"
            )
            return model
        
        # Create new dict with filtered lists
        model_dict = model.model_dump() if hasattr(model, 'model_dump') else model.dict()
        
        for field_name in list_fields:
            original_list = model_dict.get(field_name, [])
            filtered_list = [item for item in original_list if filter_fn(item)]
            model_dict[field_name] = filtered_list
            
            if len(filtered_list) < len(original_list):
                print(f"  Filtered {field_name}: {len(original_list)} → {len(filtered_list)} items")
        
        # Return new model instance
        return model.__class__(**model_dict)
    
    
    def _apply_whitelist_filter_llm(
        self,
        model: BaseModel,
        whitelist: Union[List[str], Set[str]],
        field_name: str = 'params',
        stop: list = []
    ) -> BaseModel:
        """
        Apply whitelist filtering using LLM-based matching.
        
        Uses the LLM to intelligently match extracted parameter names against
        a whitelist of valid field names, handling variations and renaming
        parameters to use the canonical whitelist names.
        
        Args:
            model: The Pydantic model instance containing extracted parameters
            whitelist: Set or list of valid parameter names
            field_name: Name of the list field to filter (default: 'params')
            stop: Stop sequences for LLM generation
            
        Returns:
            New model instance with filtered and renamed parameters
        """
        import json
        from onprem.llm.helpers import parse_json_markdown
        
        # Convert model to dict
        model_dict = model.model_dump() if hasattr(model, 'model_dump') else model.dict()
        
        # Get the list of items to filter
        items = model_dict.get(field_name, [])
        
        if not items:
            return model
        
        # Convert whitelist to sorted list for consistent display
        whitelist_list = sorted(list(whitelist))
        
        # Build the prompt
        prompt = f"""Below is a list of valid parameter names (with optional descriptions):

<begin_list>
{json.dumps(whitelist_list, indent=2)}
</end_list>

For the following list of extracted parameters, identify which parameters match the valid parameter names above. 
Return a new list containing ONLY the parameters that match valid names, with the 'name' field updated to use the exact valid parameter name from the list.

Rules:
1. Match parameters intelligently (handle abbreviations, modifiers, units, case differences)
2. Update the 'name' field to the exact valid name from the list
3. Keep all other fields (value, unit, name_of_system) unchanged
4. Exclude parameters that don't match any valid name
5. Be sure to not exclude relevant matches by accident
6. Return valid JSON format

Extracted parameters to filter:
{json.dumps(items, indent=2)}

Return the filtered list as a JSON array:"""
        
        # Call the LLM
        try:
            response_text = self.llm.prompt(prompt, stop=stop)
            
            # Use existing parse_json_markdown utility to extract and parse JSON
            filtered_items = parse_json_markdown(response_text)
            
            # Validate that it's a list
            if not isinstance(filtered_items, list):
                print(f"Warning: LLM returned non-list response, keeping original items")
                filtered_items = items
                
        except Exception as e:
            print(f"Warning: Error during LLM whitelist filtering: {e}")
            print(f"Response was: {response_text[:200] if 'response_text' in locals() else 'N/A'}...")
            print("Keeping original items")
            filtered_items = items
        
        # Update the model dict with filtered items
        model_dict[field_name] = filtered_items
        
        original_count = len(items)
        filtered_count = len(filtered_items)
        
        if filtered_count < original_count:
            print(f"  Whitelist filtered {field_name}: {original_count} → {filtered_count} items")
        
        # Return new model instance
        return model.__class__(**model_dict)
    
    
    def extract_parameters(
        self,
        fpath: Optional[str] = None,
        content: Optional[str] = None,
        parameter_whitelist: Optional[Union[List[str], set, str]] = None,
        max_length: Optional[int] = None,
        preproc_fn: Optional[Callable] = None,
        pdf_pages: List[int] = [],
        return_as: str = 'dict',
        stop: list = [],
        filter_fn: Optional[Callable[[dict], bool]] = None,
        postproc_fn: Optional[Callable] = None,
        prompt: Optional[str] = None,
        **kwargs
    ) -> Union[Any, dict, str]:
        """
        Extract system parameters from a document with intelligent name matching.
        
        This is a specialized convenience method for parameter extraction that:
        1. Uses SystemParameter/ParamCollection models automatically
        2. Uses SYSTEM_PARAMETERS_PROMPT by default
        3. Provides intelligent LLM-based matching against a parameter whitelist
        4. Handles LLM name variations (case, abbreviations, units, etc.)
        
        Args:
            fpath: Path to document file
            content: Raw text content (alternative to fpath)
            
            parameter_whitelist: Set/list of parameter names to keep, or path to file.
                               If provided, only parameters matching these names are returned.
                               Uses LLM-based intelligent matching to handle name variations.
                               Can be:
                               - Set/list: {'range', 'speed', 'weight'}
                               - Excel/CSV path: 'data_dictionary.xlsx'
                               - None: Return all extracted parameters (no filtering)
            
            max_length: Truncate document to this many characters
            preproc_fn: Preprocessing function for document text
            pdf_pages: Extract only from these PDF page numbers
            return_as: Output format - 'model', 'dict', or 'json'
            stop: Stop sequences for LLM
            
            filter_fn: Additional filter function for custom criteria
                      Applied AFTER whitelist matching
                      Example: lambda p: p['value'] > 0 and p['unit'] in ['mph', 'knots']
            
            postproc_fn: Post-processing function for ParamCollection
                        Applied AFTER filtering and whitelist matching
                        Example: deduplication, sorting, name standardization
            
            prompt: Custom prompt (overrides SYSTEM_PARAMETERS_PROMPT)
            
            **kwargs: Additional arguments passed to load_single_document
        
        Returns:
            Extracted parameters in format specified by return_as:
            - 'model': ParamCollection Pydantic model
            - 'dict': Dictionary with 'params' key
            - 'json': JSON string
        
        Examples:
            # Basic usage - extract all parameters
            >>> result = extractor.extract_parameters(fpath="specs.pdf")
            
            # With whitelist filtering from Python set
            >>> whitelist = {'range', 'cruise speed', 'weight', 'power'}
            >>> result = extractor.extract_parameters(
            ...     fpath="specs.pdf",
            ...     parameter_whitelist=whitelist
            ... )
            
            # With whitelist from Excel file
            >>> result = extractor.extract_parameters(
            ...     fpath="specs.pdf",
            ...     parameter_whitelist="data_dictionary.xlsx"
            ... )
            
            # With additional custom filtering
            >>> result = extractor.extract_parameters(
            ...     fpath="specs.pdf",
            ...     parameter_whitelist=whitelist,
            ...     filter_fn=lambda p: p['value'] > 0,  # Only positive values
            ... )
        """
        from .models import ParamCollection, SYSTEM_PARAMETERS_PROMPT
        
        # Use default prompt if not provided
        if prompt is None:
            prompt = SYSTEM_PARAMETERS_PROMPT
        
        # Load whitelist if it's a file path
        whitelist_set = None
        if parameter_whitelist is not None:
            if isinstance(parameter_whitelist, str):
                # It's a file path - load it
                from .helpers import load_parameter_whitelist
                whitelist_set = load_parameter_whitelist(parameter_whitelist, case_sensitive=False)
            else:
                # It's already a set or list
                whitelist_set = set(parameter_whitelist) if not isinstance(parameter_whitelist, set) else parameter_whitelist
        
        # Call extract_structured to get initial results
        result = self.extract_structured(
            prompt=prompt,
            pydantic_model=ParamCollection,
            fpath=fpath,
            content=content,
            max_length=max_length,
            preproc_fn=preproc_fn,
            pdf_pages=pdf_pages,
            return_as='model',  # Always get model first for processing
            stop=stop,
            filter_fn=None,  # Don't apply filter_fn yet
            postproc_fn=None,  # Don't apply postproc_fn yet
            **kwargs
        )
        
        # Apply whitelist filtering using LLM if whitelist is provided
        if whitelist_set is not None:
            result = self._apply_whitelist_filter_llm(
                model=result,
                whitelist=whitelist_set,
                field_name='params',
                stop=stop
            )
        
        # Apply custom filter function if provided
        if filter_fn is not None:
            result = self._apply_filter_fn(
                model=result,
                filter_fn=filter_fn,
                filter_field='params'
            )
        
        # Apply post-processing function if provided
        if postproc_fn is not None:
            result = postproc_fn(result)
        
        # Convert to requested format
        if return_as == 'model':
            return result
        elif return_as == 'dict':
            return result.model_dump() if hasattr(result, 'model_dump') else result.dict()
        else:  # return_as == 'json'
            return result.model_dump_json() if hasattr(result, 'model_dump_json') else result.json()

In [ ]:
show_doc(Extractor.apply)

---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/extractor.py#L32){target="_blank" style="float:right; font-size:smaller"}

### Extractor.apply

>      Extractor.apply (ex_prompt_template:str, fpath:Optional[str]=None,
>                       content:Optional[str]=None, unit:str='paragraph',
>                       preproc_fn:Optional[Callable]=None,
>                       filter_fn:Optional[Callable]=None,
>                       clean_fn:Optional[Callable]=None,
>                       pdf_pages:List[int]=[], maxchars=2048, stop:list=[],
>                       pdf_use_unstructured:bool=False, **kwargs)

*Apply the prompt to each `unit` (where a "unit" is either a paragraph or sentence) optionally filtered by `filter_fn`.
Results are stored in a `pandas.Dataframe`.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ex_prompt_template | str |  | A prompt to apply to each `unit` in document. Should have a single variable, `{text}` |
| fpath | Optional | None | A path to to a single file of interest (e.g., a PDF or MS Word document). Mutually-exclusive with `content`. |
| content | Optional | None | Text content of a document of interest.  Mutually-exclusive with `fpath`. |
| unit | str | paragraph | One of {'sentence', 'paragraph'}. |
| preproc_fn | Optional | None | Function should accept a text string and returns a new preprocessed input. |
| filter_fn | Optional | None | A function that accepts a sentence or paragraph and returns `True` if prompt should be applied to it. |
| clean_fn | Optional | None | A function that accepts a sentence or paragraph and returns "cleaned" version of the text. (applied after `filter_fn`) |
| pdf_pages | List | [] | If `fpath` is a PDF document, only apply prompt to text on page numbers listed in `pdf_pages` (starts at 1). |
| maxchars | int | 2048 | units (i.e., paragraphs or sentences) larger than `maxchars` split. |
| stop | list | [] | list of characters to trigger the LLM to stop generating. |
| pdf_use_unstructured | bool | False | If True, use unstructured package to extract text from PDF. |
| kwargs |  |  |  |

**Example:  Information Extraction With `Extractor.apply`**

In [ ]:
# | notest

from onprem import LLM
from onprem.pipelines import Extractor

In [ ]:
# | notest

prompt_template = "<s>[INST] {prompt} [/INST]" # prompt template for Mistral
llm = LLM(model_url='https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf', 
          n_gpu_layers=33,  # change based on your system
          verbose=False, mute_stream=True, 
          prompt_template=prompt_template)
extractor = Extractor(llm)

/home/amaiya/mambaforge/envs/llm/lib/python3.9/site-packages/langchain_core/language_models/llms.py:239: DeprecationWarning: callback_manager is deprecated. Please use callbacks instead.
  warnings.warn(


In [ ]:
# | notest

prompt = """Extract citations from the following sentences. Return #NA# if there are no citations in the text. Here are some examples:

[SENTENCE]:pretrained BERT text classifier (Devlin et al., 2018), models for sequence tagging (Lample et al., 2016)
[CITATIONS]:(Devlin et al., 2018), (Lample et al., 2016)
[SENTENCE]:Machine learning (ML) is a powerful tool.
[CITATIONS]:#NA#
[SENTENCE]:Following inspiration from a blog post by Rachel Thomas of fast.ai (Howard and Gugger, 2020), we refer to this as Augmented Machine Learning or AugML
[CITATIONS]:(Howard and Gugger, 2020)
[SENTENCE]:{text}
[CITATIONS]:"""

In [ ]:
# | notest

content = """
For instance, the fit_onecycle method employs a 1cycle policy (Smith, 2018). 
"""
df = extractor.apply(prompt, content=content, stop=['\n'])
assert df['Extractions'][0].strip().startswith('(Smith, 2018)')

In [ ]:
# | notest

content ="""In the case of text, this may involve language-specific preprocessing (e.g., tokenization)."""
df = extractor.apply(prompt, content=content, stop=['\n'])
assert df['Extractions'][0].strip().startswith('#NA#')

In [ ]:
show_doc(Extractor.extract_structured)

---

[source](https://github.com/amaiya/onprem/blob/master/onprem/pipelines/extractor/base.py#L106){target="_blank" style="float:right; font-size:smaller"}

### Extractor.extract_structured

```python

def extract_structured(
    prompt:str, # Prompt for structured extraction (should include {text} placeholder if needed)
    pydantic_model, # Pydantic model defining the output structure
    fpath:Optional=None, # Path to a single file. Mutually-exclusive with `content`.
    content:Optional=None, # Text content. Mutually-exclusive with `fpath`.
    max_length:Optional=None, # Maximum characters to process from document (None = no limit)
    preproc_fn:Optional=None, # Preprocessing function for document text
    pdf_pages:List=[], # If `fpath` is a PDF, only use text from these page numbers
    return_as:str='model', # One of {'model', 'dict', 'json'}. Format to return results.
    stop:list=[], # List of stop sequences for LLM
    filter_fn:Optional=None, # Filter function for extracted items (applied after extraction)
    postproc_fn:Optional=None, # Post-processing function (applied after filtering)
    filter_field:Optional=None, # Field name to filter (auto-detect if None)
    kwargs:VAR_KEYWORD
)->Union: # Extra kwargs fed to `load_single_document`


```

*Perform document-level structured extraction using Pydantic models.*

This method processes the entire document (or a large section) in a single LLM call,
using a Pydantic model to enforce structured output. This is more efficient than
chunk-by-chunk processing when you need to extract structured information that may
span multiple sections of the document.

Unlike `apply()` which processes each paragraph/sentence separately, this method:
- Makes a single LLM call for the entire document
- Maintains full document context for better relationship detection
- Enforces schema compliance via Pydantic models
- Returns structured data in your desired format

Example use cases:
- Extract all system parameters with values and units
- Extract key performance parameters with thresholds and objectives
- Extract entities, relationships, or structured specifications
- Any extraction requiring cross-document context

Args:
    prompt: Prompt template for extraction. Use {text} placeholder if you want 
           the document text inserted (recommended for clarity).
    pydantic_model: Pydantic BaseModel class defining output structure
    fpath: Path to document file (PDF, Word, text, etc.)
    content: Raw text content (alternative to fpath)
    max_length: Truncate document to this many characters (None = use full document)
    preproc_fn: Optional function to preprocess document text before extraction
    pdf_pages: For PDFs, extract only from these page numbers (1-indexed)
    return_as: Output format - 'model' (Pydantic model), 'dict', or 'json' string
    stop: Stop sequences for LLM generation
    **kwargs: Additional arguments passed to load_single_document

Returns:
    Extracted data in format specified by return_as parameter:
    - 'model': Pydantic model instance
    - 'dict': Python dictionary
    - 'json': JSON string

Example:
    ```python
    from pydantic import BaseModel, Field
    from typing import List

    class Parameter(BaseModel):
        name: str = Field(description="Parameter name")
        value: float = Field(description="Numeric value")
        unit: str = Field(description="Unit of measurement")

    class ParameterList(BaseModel):
        parameters: List[Parameter] = Field(default=[], description="All parameters found")

    extractor = Extractor(llm)
    result = extractor.extract_structured(
        prompt="Extract all technical parameters from: {text}",
        pydantic_model=ParameterList,
        fpath="specs.pdf",
        return_as='dict'
    )
    print(result['parameters'])
    ```

**Example:  Structured Information Extraction With `extract_structured`**

```python
class Parameter(BaseModel):
    name: str = Field(description="Parameter name")
    value: float = Field(description="Numeric value")
    unit: str = Field(description="Unit of measurement")

class ParameterList(BaseModel):
    parameters: List[Parameter] = Field(default=[], description="All parameters found")

extractor = Extractor(llm)
result = extractor.extract_structured(
    prompt="Extract all measured quantities from: {text}",
    pydantic_model=ParameterList,
    content="A person 6ft tall was driving 90 mph.",
    return_as='dict'
)
print(result['parameters'])

# [{'name': 'height', 'value': 6.0, 'unit': 'ft'}, {'name': 'speed', 'value': 90.0, 'unit': 'mph'}]
```

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()